In [1]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


In [2]:
cols_keep = [
 'source_id',
 'ra',
 'ra_error',
 'dec',
 'dec_error',
 'parallax',
 'parallax_error',
 'pmra',
 'pmra_error',
 'pmdec',
 'pmdec_error',
 'ra_dec_corr',
 'ra_parallax_corr',
 'ra_pmra_corr',
 'ra_pmdec_corr',
 'dec_parallax_corr',
 'dec_pmra_corr',
 'dec_pmdec_corr',
 'parallax_pmra_corr',
 'parallax_pmdec_corr',
 'pmra_pmdec_corr',
 'astrometric_gof_al',
 'astrometric_chi2_al',
 'astrometric_excess_noise',
 'astrometric_excess_noise_sig',
 'astrometric_sigma5d_max',
 'ipd_gof_harmonic_amplitude',
 'ipd_frac_multi_peak',
 'ipd_frac_odd_win',
 'ruwe',
 'phot_g_mean_flux',
 'phot_g_mean_flux_error',
 'phot_g_mean_mag',
 'phot_bp_mean_mag',
 'phot_rp_mean_mag',
 'phot_bp_rp_excess_factor',
 'radial_velocity',
 'radial_velocity_error',
 'rv_expected_sig_to_noise',
 'rvs_spec_sig_to_noise',
 'l',
 'b'
 ]

In [ ]:
dir_inp = "./data/600pc"
dir_out = "./data/600pc"

os.makedirs(dir_out, exist_ok=True)

In [4]:
def process_file(input_file, output_file, columns):
    """
    Read a large CSV file in chunks and write only selected columns
    to a lighter CSV file without loading everything into memory.
    """
    # Clean existing output file if exists
    if os.path.exists(output_file):
        os.remove(output_file)

    # Iterate through the file in chunks
    for chunk in pd.read_csv(input_file, 
                             usecols=columns, 
                             chunksize=200_000, 
                             dtype={"source_id": "string"}
                             ):
        
        # Convert source_id to int64 safely
        chunk["source_id"] = chunk["source_id"].astype("int64")

        # Safe downcast all other float64 columns
        float_cols = [
            col for col in chunk.select_dtypes(include=["float64"]).columns
            if col != "source_id"
        ]
        chunk[float_cols] = chunk[float_cols].astype("float32")

        # Append chunk to output file
        chunk.to_csv(output_file, 
                     mode="a", 
                     index=False,
                     header=not os.path.exists(output_file)
                     )

In [5]:
def process_file_to_parquet(input_file, output_file, columns):
    writer = None

    for chunk in pd.read_csv(
            input_file,
            usecols=columns,
            chunksize=200_000,
            dtype={"source_id": "string"}):
        
        # convert source_id safely
        chunk["source_id"] = chunk["source_id"].astype("int64")

        # downcast other floats
        float_cols = [
            c for c in chunk.select_dtypes(include=["float64"]).columns
            if c != "source_id"
        ]
        chunk[float_cols] = chunk[float_cols].astype("float32")

        # convert to Arrow table
        table = pa.Table.from_pandas(chunk, preserve_index=False)

        # write chunks incrementally
        if writer is None:
            writer = pq.ParquetWriter(output_file, table.schema)
        writer.write_table(table)

    if writer is not None:
        writer.close()


In [6]:
def main():
    for fname in os.listdir(dir_inp):
        if not fname.endswith(".csv"):
            continue
        
        input_path = os.path.join(dir_inp, fname)
        name_no_ext, _ = os.path.splitext(fname)
        output_path = os.path.join(dir_out, f"{name_no_ext}.parquet")

        print(f"Processing {fname} ...")

        try:
            process_file_to_parquet(input_path, output_path, cols_keep)
        except Exception as e:
            print(f"Error during processing {fname}: {e}")
            print("Original file preserved.\n")
            continue

        print(f"Lite file created: {output_path}")

        # ---- SAFE DELETION PART ----
        try:
            # Remove original file immediately
            os.remove(input_path)

            # On Linux/macOS "trash" is irrelevant; on Windows we ensure full removal
            # shutil.rmtree only for directories; for files os.remove() is enough.
            print(f"Original file removed: {input_path}\n")

        except Exception as e:
            print(f"Could not delete original file {input_path}: {e}\n")

    print("All files processed.")

In [7]:
if __name__ == "__main__":
    main()

Processing m16_16-5.csv ...
Lite file created: ./data/600pc_r/m16_16-5.parquet
Original file removed: ./data/600pc/m16_16-5.csv

Processing m11-12.csv ...
Lite file created: ./data/600pc_r/m11-12.parquet
Original file removed: ./data/600pc/m11-12.csv

All files processed.
